# 🫁 ThoraVis — Thoracic Pathology Classification
### Multi-label Chest X-ray Classification · PyTorch + HuggingFace + OpenCV

**Dataset:** NIH ChestX-ray14 (112,120 X-rays · 14 pathology labels)  
**Model:** ViT-B/16 fine-tuned via HuggingFace Transformers  
**Preprocessing:** Clinical-grade OpenCV pipeline (CLAHE · Bilateral · Sobel)  
**Metric:** Per-class AUC-ROC + Grad-CAM visualization

---

## Contents
1. Environment Setup
2. Dataset Exploration (HuggingFace)
3. OpenCV Preprocessing Pipeline
4. Model Architecture
5. Training
6. Evaluation & AUC-ROC
7. Grad-CAM Visualization


## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install -q torch torchvision transformers datasets opencv-python \
               scikit-learn matplotlib tqdm Pillow

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from datasets import load_dataset

# Check device
if torch.cuda.is_available():     DEVICE = 'cuda'
elif torch.backends.mps.is_available(): DEVICE = 'mps'
else:                              DEVICE = 'cpu'
print(f'Using device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')
print(f'OpenCV version: {cv2.__version__}')

## 2. Dataset Exploration — NIH ChestX-ray14 via HuggingFace

In [ ]:
from datasets import load_dataset

# Stream a small slice for exploration (avoids downloading full 40GB)
print('Loading NIH ChestX-ray14 from HuggingFace...')
ds = load_dataset(
    'alkzar90/NIH-Chest-X-ray-dataset',
    'image-classification',
    split='train[:200]',
    trust_remote_code=True,
)
print(f'Loaded {len(ds):,} samples for exploration')
print(f'Features: {ds.features}')

In [ ]:
PATHOLOGY_LABELS = [
    'No Finding', 'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration',
    'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation',
    'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia'
]

# ── Label frequency analysis ──────────────────────────────────────────────────
from collections import Counter

label_counts = Counter()
for sample in ds:
    for lbl in sample['labels']:
        label_counts[PATHOLOGY_LABELS[lbl]] += 1

labels_sorted = sorted(label_counts.items(), key=lambda x: -x[1])
names, counts = zip(*labels_sorted)

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.barh(names, counts, color=plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(names))))
ax.set_xlabel('Count (in 200-sample slice)')
ax.set_title('NIH ChestX-ray14 — Label Distribution (Exploration Slice)', fontsize=12)
for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            str(count), va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../results/label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Note: "No Finding" dominates (~53% of full dataset) → class imbalance is handled with inverse-frequency weighting.')

In [ ]:
# ── Sample X-ray grid ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('NIH ChestX-ray14 — Sample Images', fontsize=13)

for i, ax in enumerate(axes.flatten()):
    sample = ds[i]
    img = np.array(sample['image'])
    lbls = [PATHOLOGY_LABELS[l] for l in sample['labels']]
    ax.imshow(img, cmap='gray')
    ax.set_title('\n'.join(lbls[:2]), fontsize=7)
    ax.axis('off')

plt.tight_layout()
plt.savefig('../results/sample_grid.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. OpenCV Preprocessing Pipeline

In [ ]:
from src.preprocessing import XRayPreprocessor

preprocessor = XRayPreprocessor(augment=False)

# Demonstrate the full preprocessing pipeline on a real X-ray
raw_pil  = ds[5]['image']
raw_np   = np.array(raw_pil)
labels_5 = [PATHOLOGY_LABELS[l] for l in ds[5]['labels']]
print(f'Sample labels: {labels_5}')

preprocessor.visualize_pipeline(
    raw_np,
    save_path='../results/preprocessing_pipeline.png'
)

In [ ]:
# ── CLAHE deep-dive ───────────────────────────────────────────────────────────
# Show effect of different CLAHE clip limits

gray = cv2.cvtColor(raw_np, cv2.COLOR_RGB2GRAY) if raw_np.ndim == 3 else raw_np
gray = cv2.resize(gray, (224, 224))

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
fig.suptitle('CLAHE Clip Limit Comparison (OpenCV)', fontsize=12)

axes[0].imshow(gray, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')

for i, clip in enumerate([1.0, 3.0, 8.0]):
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)
    axes[i+1].imshow(enhanced, cmap='gray')
    axes[i+1].set_title(f'CLAHE clip={clip}')
    axes[i+1].axis('off')

plt.tight_layout()
plt.savefig('../results/clahe_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('clip=3.0 balances fine detail vs. artifact suppression — used in ThoraVis.')

## 4. Model Architecture — ViT-B/16 + Custom Head

In [ ]:
from src.model import ThoraVisClassifier

model = ThoraVisClassifier(freeze_backbone=True)
model = model.to(DEVICE)

param_info = model.count_parameters()
print('\n── ThoraVisClassifier Parameter Count ───────────────────────')
for k, v in param_info.items():
    print(f'  {k:<14}: {v:>12,}')

# Quick forward pass test
dummy_input = torch.randn(2, 3, 224, 224).to(DEVICE)
with torch.no_grad():
    logits = model(dummy_input)
print(f'\nForward pass: input {tuple(dummy_input.shape)} → logits {tuple(logits.shape)}')
print(f'Probabilities (dummy): {torch.sigmoid(logits)[0].round(decimals=2).tolist()}')

In [ ]:
# ── Architecture diagram ──────────────────────────────────────────────────────
print('''
ThoraVis Architecture:
─────────────────────────────────────────────────────────────────
INPUT: Chest X-ray (224×224 PNG, grayscale)
  │
  ▼  [OpenCV Preprocessing]
  │  CLAHE → Bilateral Filter → RGB conversion → ImageNet normalization
  │
  ▼  [ViT-B/16 Backbone — google/vit-base-patch16-224-in21k]
  │  Splits image into 14×14 = 196 patches (16×16 px each)
  │  12 transformer encoder layers, hidden=768, heads=12
  │  → CLS token: (B, 768)
  │
  ▼  [Classification Head]
  │  LayerNorm(768)
  │  → Linear(768 → 256) → GELU → Dropout(0.3)
  │  → Linear(256 → 14)
  │
  ▼  [Loss: WeightedBCEWithLogitsLoss]
  │  Inverse-frequency class weights for imbalance
  │
  ▼  OUTPUT: 14-dim sigmoid probabilities
     One per pathology: Atelectasis, Cardiomegaly, ...
─────────────────────────────────────────────────────────────────
''')

## 5. Training

In [ ]:
from src.train import ThoraVisTrainer

# ── Configure training run ────────────────────────────────────────────────────
# For notebook demo: 2,000 images, 3 epochs (~15min on GPU, ~45min on CPU)
# For full training: set subset_size=None, epochs=20

trainer = ThoraVisTrainer(
    subset_size=2000,      # set to None for full dataset
    epochs=3,
    batch_size=16,
    lr=2e-5,
    warmup_epochs=1,
    checkpoint_dir='../models',
)

history = trainer.train()
trainer.print_per_class_auc()

In [ ]:
from src.evaluate import plot_training_history

plot_training_history(history, save_path='../results/training_history.png')

## 6. Evaluation — Per-class AUC-ROC

In [ ]:
import torch
from src.evaluate import collect_predictions, compute_auc_table, print_auc_table, plot_roc_curves

device = torch.device(DEVICE)

# Load best checkpoint
ckpt = torch.load('../models/best_thoravis.pt', map_location=device)
trainer.model.load_state_dict(ckpt['state_dict'])
print(f"Loaded checkpoint from epoch {ckpt['epoch']} (val AUC={ckpt['val_auc']:.4f})")

# Collect predictions on test set
probs, labels = collect_predictions(trainer.model, trainer.test_loader, device)
print(f'Test set: {probs.shape[0]} samples')

# AUC-ROC table
auc_dict = compute_auc_table(probs, labels)
print_auc_table(auc_dict)

In [ ]:
plot_roc_curves(probs, labels, save_path='../results/roc_curves.png')

In [ ]:
from src.evaluate import compute_pr_f1
import pandas as pd

pr_f1 = compute_pr_f1(probs, labels, threshold=0.5)
df = pd.DataFrame(pr_f1).T.round(3)
df.index.name = 'Pathology'
print('\n── Precision / Recall / F1 @ threshold=0.5 ─────────────────')
print(df.sort_values('f1', ascending=False).to_string())

## 7. Grad-CAM — What Did the Model See?

In [ ]:
from src.gradcam import GradCAMViT
from src.preprocessing import XRayPreprocessor

# Load a test sample
ds_test = load_dataset(
    'alkzar90/NIH-Chest-X-ray-dataset',
    'image-classification',
    split='test[:50]',
    trust_remote_code=True,
)

# Find a sample with a confirmed finding
sample_idx = next(
    i for i, s in enumerate(ds_test)
    if any(l != 0 for l in s['labels'])   # skip 'No Finding'
)
sample     = ds_test[sample_idx]
pil_img    = sample['image']
true_labels = [PATHOLOGY_LABELS[l] for l in sample['labels']]
print(f'Ground truth: {true_labels}')

# Preprocess
prep           = XRayPreprocessor(augment=False)
image_tensor   = prep.preprocess_pil(pil_img).to(device)

# Grad-CAM
gradcam = GradCAMViT(trainer.model, device=device)
gradcam.visualize(
    original_pil_image=pil_img,
    image_tensor=image_tensor,
    top_k=3,
    save_path='../results/gradcam_visualization.png',
)

In [ ]:
# ── Single heatmap deep-dive ──────────────────────────────────────────────────
from src.preprocessing import overlay_heatmap

target_cls = sample['labels'][0] if sample['labels'] else 2  # default: Cardiomegaly
heatmap    = gradcam.generate(image_tensor, target_class=target_cls)

orig_np = np.array(pil_img.resize((224, 224)))
if orig_np.ndim == 2:
    orig_np = cv2.cvtColor(orig_np, cv2.COLOR_GRAY2RGB)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle(f'Grad-CAM Deep Dive: {PATHOLOGY_LABELS[target_cls]}', fontsize=12)

ax1.imshow(orig_np, cmap='gray'); ax1.set_title('Original X-ray'); ax1.axis('off')
ax2.imshow(heatmap, cmap='jet');  ax2.set_title('Grad-CAM Heatmap'); ax2.axis('off')
blended = overlay_heatmap(orig_np, heatmap)
ax3.imshow(blended);              ax3.set_title('Overlay (α=0.45)'); ax3.axis('off')

plt.tight_layout()
plt.savefig(f'../results/gradcam_{PATHOLOGY_LABELS[target_cls].lower()}.png',
            dpi=150, bbox_inches='tight')
plt.show()

---
## Summary

| Component | Implementation | Library |
|---|---|---|
| Dataset loading | `alkzar90/NIH-Chest-X-ray-dataset` streaming | HuggingFace `datasets` |
| ViT backbone | `google/vit-base-patch16-224-in21k` | HuggingFace `transformers` |
| Preprocessing | CLAHE, bilateral filter, Sobel edges | OpenCV |
| Heatmap overlay | Grad-CAM + `cv2.applyColorMap` | OpenCV + PyTorch |
| Training loop | AdamW + CosineAnnealingLR + grad clipping | PyTorch |
| Evaluation | Per-class AUC-ROC, PR-F1 | scikit-learn |

**Best Val AUC achieved on 2k-sample demo run** → varies by run, typical range 0.70–0.85  
**Full dataset (112k images, 20 epochs)** → expected macro AUC ≥ 0.80 on major pathologies

> *This notebook demonstrates end-to-end medical imaging ML: data loading → preprocessing → model architecture → training → evaluation → interpretability.*
